# SQLAlchemy: working with relational databases

Customers and orders need persistent relationships and a reliable spending report. Define keys and constraints before writing rows, group related writes into transactions, and check that customers without orders remain visible when the question concerns the whole customer base.

A database stores related records and lets us query them together. SQLAlchemy provides SQL expressions through its Core interface and maps rows to Python objects through its ORM.

We will build a small café database, query customer spending, demonstrate transaction rollback, and use an ORM model. All records are **invented teaching data**. SQLite is included with Python, so no server or account is needed. Run cells in order. Rerun from the engine setup to reset the in-memory database.

## 1. Install and import

If needed, run `%pip install "sqlalchemy>=2.0"` in a separate cell. These examples use the SQLAlchemy 2.x interface.

| Component | Responsibility |
| --- | --- |
| Engine | Database connection configuration and pooling |
| Connection | Execute Core statements |
| Table and MetaData | Describe database schema |
| Session | Track ORM objects and coordinate their database operations |
| Transaction | Commit related writes together or roll them back |

SQLAlchemy is a toolkit, not a database. SQLite stores and executes the data operations here.

In [ ]:
from pathlib import Path
import csv
import json
import sqlalchemy as sa
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, ForeignKey, CheckConstraint
from sqlalchemy import select, insert, update, delete, func, text, event
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

print("SQLAlchemy:", sa.__version__)

## 2. Create an in-memory SQLite engine

The database lives only for this engine's connection lifetime; it is not a persistent file. This keeps the learning examples isolated from existing databases. Dispose the old engine when rerunning setup.

SQLite foreign-key enforcement must be enabled for each connection. An event listener does that before our tables are used. The notebook runs in a single thread; in-memory connection sharing across threads requires additional configuration.

In [ ]:
if "engine" in globals():
    engine.dispose()
engine = create_engine("sqlite+pysqlite:///:memory:")

@event.listens_for(engine, "connect")
def enable_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()

with engine.connect() as connection:
    print("Foreign keys enabled:", connection.scalar(text("PRAGMA foreign_keys")))

## 3. Define relationships that prevent ambiguous orders

Identify the primary and foreign keys before creating tables. Ask whether two customers may share a name and why the name is not the relationship key. Predict which invalid records the declared constraints should reject.

Each customer has a primary key. Each order references a customer through a foreign key. Email is required and unique. Order amounts are stored as integer cents to avoid binary floating-point currency rounding.

A check constraint rejects negative amounts. `create_all` creates absent tables; it does not migrate an existing schema when definitions change.

In [ ]:
metadata = MetaData()
customers = Table("customers", metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True))
orders = Table("orders", metadata,
    Column("id", Integer, primary_key=True),
    Column("customer_id", ForeignKey("customers.id"), nullable=False),
    Column("amount_cents", Integer, nullable=False),
    CheckConstraint("amount_cents >= 0", name="nonnegative_amount"))
metadata.create_all(engine)
print("Tables:", sa.inspect(engine).get_table_names())

## 4. Insert records in one transaction

`engine.begin()` opens a connection and transaction. Leaving the block normally commits its writes; an exception rolls them back. Insert parents before rows that reference them.

Passing a list of dictionaries inserts several rows. Running this seed cell twice without resetting the database fails on duplicate primary keys, which protects existing records from accidental replacement.

In [ ]:
with engine.begin() as connection:
    connection.execute(insert(customers), [
        {"id": 1, "name": "Asha", "email": "asha@example.com"},
        {"id": 2, "name": "Ben", "email": "ben@example.com"},
        {"id": 3, "name": "Cleo", "email": "cleo@example.com"},
    ])
    connection.execute(insert(orders), [
        {"id": 1, "customer_id": 1, "amount_cents": 1200},
        {"id": 2, "customer_id": 1, "amount_cents": 800},
        {"id": 3, "customer_id": 2, "amount_cents": 1500},
    ])
print("Inserted three customers and three orders.")

## 5. Select, filter, and order results

`select` constructs a SQL statement; executing it retrieves results. Use an explicit `order_by` whenever row order matters. A `where` clause limits which rows qualify.

`mappings()` provides dictionary-like rows; converting them to dictionaries produces ordinary Python records. Fetch results while the connection is open.

In [ ]:
statement = select(customers.c.id, customers.c.name).where(customers.c.id >= 2).order_by(customers.c.id)
print(statement)
with engine.connect() as connection:
    selected_customers = [dict(row) for row in connection.execute(statement).mappings()]
print(selected_customers)

## 6. Bind values instead of inserting them into SQL text

SQL expressions bind values for you. When using textual SQL, named parameters such as `:name` keep values separate from SQL syntax. Do not assemble query values with string interpolation.

Parameters represent values, not table names or SQL keywords. Choose identifiers through known schema objects or validated application choices.

In [ ]:
with engine.connect() as connection:
    named_customer = connection.execute(
        text("SELECT id, name FROM customers WHERE name = :name"), {"name": "Asha"}
    ).mappings().one()
    unusual_name = "Asha' OR 1=1 --"
    literal_matches = connection.execute(
        select(customers.c.id).where(customers.c.name == unusual_name)
    ).scalars().all()
print(dict(named_customer))
print("Matches for the literal unusual name:", literal_matches)

## 7. Join orders to customers

A join combines rows through a relationship. An inner join returns customers with matching orders, once per order. That means a customer's name can appear more than once.

SQLAlchemy can infer this join from the foreign key, but we supply the condition explicitly to make the relationship visible.

In [ ]:
joined_query = select(customers.c.name, orders.c.id.label("order_id"), orders.c.amount_cents).join_from(
    customers, orders, customers.c.id == orders.c.customer_id
).order_by(orders.c.id)
with engine.connect() as connection:
    joined_records = [dict(row) for row in connection.execute(joined_query).mappings()]
print(joined_records)

## 8. Keep customers who have not placed an order

Predict the output for a customer with no orders. Inspect how the outer join, count expression and missing total interact. COUNT(*) after a left join can count the unmatched placeholder row, so choose the count deliberately.

A left outer join keeps every customer. Count the order ID, not `count(*)`: the unmatched customer still produces a joined row, but its order ID is NULL.

`sum` over no matching amounts is NULL, so `coalesce` replaces it with zero. Group by both the customer's ID and name for a clear, portable grouping.

In [ ]:
spending_query = (
    select(customers.c.id, customers.c.name,
           func.count(orders.c.id).label("order_count"),
           func.coalesce(func.sum(orders.c.amount_cents), 0).label("total_cents"))
    .select_from(customers.outerjoin(orders, customers.c.id == orders.c.customer_id))
    .group_by(customers.c.id, customers.c.name)
    .order_by(customers.c.id)
)
with engine.connect() as connection:
    spending_records = [dict(row) for row in connection.execute(spending_query).mappings()]
print(spending_records)

## 9. Update and delete specific records

Use a deliberate `where` clause to target writes. An update or delete without one can affect every row in the table.

Add a temporary customer, rename it, and remove it in one transaction. The original three customers remain. For parent rows with existing orders, the foreign-key constraint prevents deletion unless a suitable relationship policy is defined.

In [ ]:
with engine.begin() as connection:
    connection.execute(insert(customers), {"id": 99, "name": "Temporary", "email": "temporary@example.com"})
    connection.execute(update(customers).where(customers.c.id == 99).values(name="Temporary revised"))
    revised_name = connection.scalar(select(customers.c.name).where(customers.c.id == 99))
    connection.execute(delete(customers).where(customers.c.id == 99))
print("Name before deletion:", revised_name)

## 10. Undo all related writes when one step fails

Predict which rows should exist after the deliberate failure. Inspect the database afterward rather than relying only on the exception message. The intended invariant is an all-or-nothing transaction.

Both statements below belong to one transaction. The first insert is valid; the second repeats an existing unique email. The resulting exception causes the entire transaction to roll back, including the first insert.

Catch the exception outside the transaction block so rollback completes before issuing another query.

In [ ]:
rollback_observed = False
try:
    with engine.begin() as connection:
        connection.execute(insert(customers), {"id": 90, "name": "Rolled back", "email": "rollback@example.com"})
        connection.execute(insert(customers), {"id": 91, "name": "Duplicate email", "email": "asha@example.com"})
except IntegrityError:
    rollback_observed = True
    print("Expected unique-constraint failure; transaction rolled back.")
with engine.connect() as connection:
    remaining_temporary = connection.scalar(select(func.count()).select_from(customers).where(customers.c.id.in_([90, 91])))
print("Rows left from the failed transaction:", remaining_temporary)

## 11. Check database constraints

The database enforces relationships and valid values even if the application forgets to check them. These two separate transactions intentionally attempt an unknown customer reference and a negative amount.

Each failure is caught after its transaction rolls back. No invalid order should remain.

In [ ]:
constraint_failures = []
for label, row in [
    ("foreign key", {"id": 90, "customer_id": 999, "amount_cents": 500}),
    ("nonnegative amount", {"id": 91, "customer_id": 1, "amount_cents": -500}),
]:
    try:
        with engine.begin() as connection:
            connection.execute(insert(orders), row)
    except IntegrityError:
        constraint_failures.append(label)
        print("Expected constraint failure:", label)

## 12. Map a Python class with the ORM

A mapped class associates attributes with table columns. Here the `Product` class uses a separate products table in the same database. Core tables and ORM models can coexist.

A Session tracks objects. `flush` sends pending changes to the database, which can assign a generated ID; it does not commit the transaction. `Session.begin()` commits on success. Closing a Session alone does not imply a commit.

In [ ]:
class Base(DeclarativeBase):
    pass

class Product(Base):
    __tablename__ = "products"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100), unique=True)
    price_cents: Mapped[int] = mapped_column(Integer)
    __table_args__ = (CheckConstraint("price_cents >= 0", name="product_nonnegative_price"),)

Base.metadata.create_all(engine)
with Session(engine) as session, session.begin():
    tea = Product(name="Tea", price_cents=300)
    session.add_all([tea, Product(name="Coffee", price_cents=400)])
    session.flush()
    tea_id = tea.id
    print("Assigned tea ID before commit:", tea_id)

## 13. Query and edit ORM objects

`session.scalars(select(Product))` yields Product objects. `session.get` retrieves by primary key and returns `None` if absent. A loaded object's changed attributes are tracked and written when the session flushes or commits.

Read attributes inside the Session. Objects can expire after commit, and later attribute access may require a database connection.

In [ ]:
with Session(engine) as session, session.begin():
    tea = session.get(Product, tea_id)
    if tea is None:
        raise LookupError("Tea product is missing")
    tea.price_cents = 350

with Session(engine) as session:
    product_records = [{"id": product.id, "name": product.name, "price_cents": product.price_cents}
                       for product in session.scalars(select(Product).order_by(Product.id))]
print(product_records)

## 14. Read result shapes deliberately

A query can return rows, mappings, or scalar values. `scalar_one()` requires exactly one row and returns its first column. `scalar_one_or_none()` allows an absent row but still rejects multiple rows.

Use these methods to express expectations instead of silently picking the first result when uniqueness matters.

In [ ]:
with engine.connect() as connection:
    customer_count = connection.execute(select(func.count()).select_from(customers)).scalar_one()
    absent_name = connection.execute(select(customers.c.name).where(customers.c.id == 999)).scalar_one_or_none()
    ordered_names = connection.execute(select(customers.c.name).order_by(customers.c.id)).scalars().all()
print("Customer count:", customer_count)
print("Missing customer's name:", absent_name)
print("Names:", ordered_names)

## 15. Make the customer report reconcile with its source

State whether the report includes every customer or only buyers. Check one customer total against their orders and verify the export’s row count. A database abstraction does not choose the business population for you.

Query the current database and export ordinary dictionaries to JSON and CSV. Keep the amount column explicitly labeled in cents, and format dollars only for display.

This cell creates `outputs` and replaces these named reports. The database itself remains in memory and will disappear when its connections are disposed. A file-backed SQLite database would use a URL such as `sqlite+pysqlite:///cafe.sqlite3`; do not change to an existing database without reviewing the example writes.

In [ ]:
with engine.connect() as connection:
    report_records = [dict(row) for row in connection.execute(spending_query).mappings()]
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
json_path = output_dir / "sqlalchemy_customer_spending.json"
csv_path = output_dir / "sqlalchemy_customer_spending.csv"
json_path.write_text(json.dumps(report_records, indent=2) + "\n", encoding="utf-8")
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "name", "order_count", "total_cents"])
    writer.writeheader()
    writer.writerows(report_records)
for record in report_records:
    print(f"{record['name']}: {record['order_count']} orders, ${record['total_cents'] / 100:.2f}")
print("Saved:", json_path)
print("Saved:", csv_path)

## 16. Release resources

Context managers above close connections and sessions after use. Disposing the engine releases its pooled connections. Because this database is in memory, reset and seed it again before running more queries.

In [ ]:
engine.dispose()
print("In-memory database connections released.")

## 17. Practice and troubleshooting

1. Add a fourth customer with no orders and confirm the report includes a zero total.
2. Add another order for Ben and verify the grouped count and total.
3. Try deleting Asha while her orders still exist. Explain the foreign-key failure.
4. Add a filter that returns only customers whose summed spending exceeds 1500 cents; use `having` for the aggregate condition.

| Symptom | What to check |
| --- | --- |
| Writes disappear after the block | Use an explicit transaction that commits; closing a connection alone does not commit |
| Duplicate rows appear after joining | Understand the one-to-many relationship before aggregating |
| Customers without orders vanish | Use an outer join and avoid filters that eliminate its NULL rows |
| SQLAlchemy rejects a raw SQL string | Wrap textual SQL with `text()` and bind its values |
| Objects cannot load attributes after Session closure | Materialize required values while the Session is open |
| Changing a model does not update a table | `create_all` is not a schema migration tool |
| A Session fails after an integrity error | Roll back or end the failed transaction before reusing it |

The official [unified tutorial](https://docs.sqlalchemy.org/en/20/tutorial/), [SELECT guide](https://docs.sqlalchemy.org/en/20/tutorial/data_select.html), [connection guide](https://docs.sqlalchemy.org/en/20/core/connections.html), and [Session basics](https://docs.sqlalchemy.org/en/20/orm/session_basics.html) provide further examples.

## Explain what the evidence supports

Describe a failed multi-step update and the state that should remain afterward. Then explain how the spending report treats a customer without orders.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: ingest retried payment events transactionally

A payment export contains retries, invalid amounts, malformed dates and unknown customer keys. Refunds are signed negative amounts, so deleting every negative amount would inflate net receipts. Reconcile 6,060 raw rows into 60 retries, 50 rejected events and 5,950 usable events. These are generated operational-style records, not actual customer payments. The generator's clean source is used only as a verification oracle, never to fill invalid source fields. Work within a small laptop-sized budget, then identify which intermediate tables or buffers would grow at scale.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through exactly what idempotency means

Submitting the same event ID twice must not create two charges. A database primary key enforces uniqueness, but blindly ignoring conflicts can hide changed payloads. In this fixture repeated IDs have identical fields; verify that fact before using conflict-do-nothing ingestion. Real conflicting retries need a documented reconciliation policy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(91)
p2_n=6000
p2_source=pd.DataFrame({'event_id':np.arange(p2_n),'customer_id':p2_rng.integers(1,121,p2_n),
    'day':pd.Timestamp('2025-01-01')+pd.to_timedelta(p2_rng.integers(0,90,p2_n),unit='D'),
    'amount_cents':p2_rng.integers(100,100000,p2_n)})
p2_source.loc[p2_source.event_id%17==0,'amount_cents'] *= -1 # Signed refunds are valid.
p2_raw=p2_source.astype(str)
p2_raw.loc[0:19,'amount_cents']='bad'
p2_raw.loc[20:29,'amount_cents']=''
p2_raw.loc[30:37,'day']='not-a-date'
p2_raw.loc[38:49,'customer_id']='9999'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-60:]],ignore_index=True)
p2_customers=pd.DataFrame({'customer_id':np.arange(1,121),'region':np.tile(['East','West','North','South'],30)})
p2_raw.to_csv(p2_out/'ledger_raw.csv',index=False)
# Independent generator-based expected population; never repair a bad value from this oracle.
p2_expected=p2_source.iloc[50:]
p2_expected_total=int(p2_expected.amount_cents.sum())
assert len(p2_raw)==6060 and len(p2_expected)==5950

from sqlalchemy import create_engine,MetaData,Table,Column,Integer,String,select,func
from sqlalchemy.dialects.sqlite import insert as p2_sqlite_insert
p2_u=p2_raw.drop_duplicates().copy()
assert p2_u.event_id.is_unique
p2_u['cents']=pd.to_numeric(p2_u.amount_cents,errors='coerce');p2_u['parsed_day']=pd.to_datetime(p2_u.day,errors='coerce')
p2_u['customer_id']=pd.to_numeric(p2_u.customer_id,errors='coerce')
p2_ok=p2_u.cents.notna()&p2_u.parsed_day.notna()&p2_u.customer_id.isin(p2_customers.customer_id)
p2_rows=[{'event_id':int(r.event_id),'customer_id':int(r.customer_id),'cents':int(r.cents)} for r in p2_u[p2_ok].itertuples()]
p2_engine=create_engine('sqlite:///:memory:');p2_meta=MetaData()
p2_events=Table('events',p2_meta,Column('event_id',Integer,primary_key=True),Column('customer_id',Integer,nullable=False),Column('cents',Integer,nullable=False))
p2_meta.create_all(p2_engine)
with p2_engine.begin() as p2_conn:
    for p2_offset in range(0,len(p2_rows),500):
        p2_conn.execute(p2_sqlite_insert(p2_events).on_conflict_do_nothing(index_elements=['event_id']),p2_rows[p2_offset:p2_offset+500])
    p2_conn.execute(p2_sqlite_insert(p2_events).on_conflict_do_nothing(index_elements=['event_id']),p2_rows[-60:])
with p2_engine.connect() as p2_conn:
    assert p2_conn.scalar(select(func.count()).select_from(p2_events))==5950
    assert p2_conn.scalar(select(func.sum(p2_events.c.cents)))==p2_expected_total

### Work through a failed transaction and a query budget

Stage a valid new event followed by a duplicate primary key inside one transaction. Catch the integrity error outside the context manager so the earlier insert rolls back. For reporting, compare issuing 120 per-customer aggregate queries with a single grouped query. Count statements as well as timing: round trips become important when the database is remote.

In [ ]:
from sqlalchemy.exc import IntegrityError
try:
    with p2_engine.begin() as p2_conn:
        p2_conn.execute(p2_events.insert(),{'event_id':999999,'customer_id':1,'cents':123})
        p2_conn.execute(p2_events.insert(),p2_rows[0])
except IntegrityError:print('Batch failed; both writes were rolled back.')
with p2_engine.connect() as p2_conn:
    assert p2_conn.scalar(select(func.count()).where(p2_events.c.event_id==999999))==0
    p2_t=time.perf_counter()
    p2_many={i:p2_conn.scalar(select(func.sum(p2_events.c.cents)).where(p2_events.c.customer_id==i)) or 0 for i in range(1,121)}
    p2_many_time=time.perf_counter()-p2_t
    p2_t=time.perf_counter()
    p2_grouped=dict(p2_conn.execute(select(p2_events.c.customer_id,func.sum(p2_events.c.cents)).group_by(p2_events.c.customer_id)).all())
    p2_group_time=time.perf_counter()-p2_t
assert p2_many==p2_grouped and sum(p2_grouped.values())==p2_expected_total
pd.DataFrame(p2_grouped.items(),columns=['customer_id','net_cents']).to_csv(p2_out/'sqlalchemy_customer_totals.csv',index=False)
print({'per_customer_statements':120,'grouped_statements':1,'many_seconds':p2_many_time,'grouped_seconds':p2_group_time})
p2_engine.dispose()

### Explain the tradeoff

A primary key prevents duplicate storage; the pre-ingestion comparison prevents silently accepting conflicting duplicate meanings. Batches of 500 bound parameter construction, while the transaction preserves all-or-nothing behavior for the chosen batch. A remote database may amplify statement overhead, but these SQLite timings are local observations only.

**Try:** change the cents on a retried ID. **Worked reasoning:** exact-row deduplication now leaves two rows sharing an ID, so the uniqueness assertion stops ingestion. Replacing it with an unconditional ignore would conceal a conflict; replacing it with an unconditional update could overwrite a legitimate historical event.